# Zomato restaurant market analysis: walkthrough

This notebook replaces `00_original_colab_eda.ipynb`. The logic lives in the tested
`zomato_analytics` package, and the notebook only tells the story. Run the whole
thing in one go with `zomato-analytics run`.

**Business questions**
1. Where is the platform's footprint, and how do markets compare in GBP?
2. How does the **UK** estate compare with the rest of the platform?
3. What is associated with a higher rating: price, table booking, online delivery?
4. Which cuisines and cities show demand that supply isn't meeting?

> Uses `data/raw/zomato.csv` if it exists. Otherwise it falls back to **synthetic demo
> data**, whose numbers are illustrative only.

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

from zomato_analytics import (
    cleaning,
    config,
    features,
    io,
    metrics,
    quality,
    sample_data,
    stats,
    warehouse,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

SOURCE = config.DEFAULT_RAW_FILE
IS_SAMPLE = not SOURCE.exists()
if IS_SAMPLE:
    SOURCE = sample_data.write_sample(config.RAW_DIR / "zomato_sample.csv")
    display(Markdown("**Using synthetic demo data: numbers are illustrative only.**"))
SOURCE

## 1. Extract and profile the raw file
Profile the source before changing anything: what would we need to tell the data owner?

In [ ]:
raw = io.read_raw(SOURCE)
print(raw.shape)
quality.profile_raw(raw)

## 2. Clean, enrich and validate
Every change is logged. The hard checks stop here if the cleaned data is wrong.

In [ ]:
df, clog = cleaning.clean(raw)
df = features.add_features(df)
display(clog.to_frame())
quality.assert_clean(df)

**Why this matters:** the original notebook averaged ratings that included the
`0.0 = not rated` placeholder, and it compared costs across 12 currencies. Here's the effect:

In [ ]:
naive_mean = pd.to_numeric(raw["Aggregate rating"]).mean()
print(f"Naive mean rating (0.0 treated as a score): {naive_mean:.2f}")
print(
    f"Mean rating of rated restaurants:            {df.loc[df.is_rated, 'aggregate_rating'].mean():.2f}"
)
print(
    f"Naive median cost (mixed currencies):        {pd.to_numeric(raw['Average Cost for two']).median():,.0f}"
)
print(
    f"Median cost for two in GBP:                  £{df.loc[~df.is_cost_outlier, 'cost_for_two_gbp'].median():.2f}"
)

## 3. Headline KPIs and market overview

In [ ]:
display(pd.Series(metrics.headline_kpis(df), name="value").to_frame())
metrics.market_overview(df)

## 4. UK spotlight

In [ ]:
display(metrics.uk_benchmark(df))
metrics.uk_city_summary(df)

## 5. What is associated with a higher rating?

Because online delivery is mostly offered in India and booking clusters in expensive
restaurants, the service effect is shown **within each price band**, and the tests are repeated India-only.

In [ ]:
display(metrics.price_band_summary(df))
metrics.service_impact(df)

In [ ]:
stats.run_all(df)[
    ["scope", "test", "n", "effect_size", "effect_magnitude", "p_value", "interpretation"]
]

## 6. Cuisines and the opportunity matrix

In [ ]:
display(metrics.cuisine_performance(df).head(15))
metrics.opportunity_matrix(df).head(15)

## 7. The same questions in SQL (DuckDB star schema)
The SQL layer is a second, independent implementation. The test suite checks that the two agree.

In [ ]:
con = warehouse.build(df)
sql = warehouse.run_analysis(con)
for name in ("02_city_rank_within_country", "03_top_cuisine_per_city", "06_cuisine_votes_pareto"):
    display(Markdown(f"**{name}**"))
    display(sql[name].head(10))

## 8. Charts

In [ ]:
from zomato_analytics import viz

tables = {
    "market_overview": metrics.market_overview(df),
    "service_impact": metrics.service_impact(df),
    "cuisine_performance": metrics.cuisine_performance(df),
    "uk_city_summary": metrics.uk_city_summary(df),
}
fig_dir = config.REPORTS_DIR / "figures"
for path in viz.render_all(df, tables, config.DEFAULT_CONFIG.price_band_labels, fig_dir):
    display(Image(filename=str(path)))

## 9. Conclusions

Run `zomato-analytics run` to produce `reports/executive_summary.md`. Its findings
and recommendations are generated from the numbers above, so they're always in step
with the data. See `docs/methodology.md` for the assumptions and limitations.